<a href="https://colab.research.google.com/github/abhinayapravallika/Content-Generator/blob/main/Copy_of_style_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import tensorflow as tf
# Load compressed models from tensorflow_hub
os.environ['TFHUB_MODEL_LOAD_FORMAT'] = 'COMPRESSED'   #This line sets an environment variable to load compressed models from TensorFlow Hub.
import IPython.display as display

import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.figsize'] = (12, 12)
mpl.rcParams['axes.grid'] = False
import numpy as np
import PIL.Image
import time
import functools
def tensor_to_image(tensor):
  tensor = tensor*255  #Scaling the tensor values to the range [0, 255]
  tensor = np.array(tensor, dtype=np.uint8) ## Converting the tensor to a NumPy array with dtype uint8 (8-bit unsigned integers)

  if np.ndim(tensor)>3:    ## Checking if the tensor has more than 3 dimensions (e.g., if it's a batch of images)
    assert tensor.shape[0] == 1 #  # Asserting that the batch size is 1
    tensor = tensor[0]   # Taking the first (and only) image from the batch
  return PIL.Image.fromarray(tensor)
content_path = tf.keras.utils.get_file("/content/content2.png")
style_path = tf.keras.utils.get_file('/content/style2.png')

ValueError: Please specify the "origin" argument (URL of the file to download).

In [ ]:
def load_img(path_to_img):
  max_dim = 512
  img = tf.io.read_file(path_to_img)
  img = tf.image.decode_image(img, channels=3)
  img = tf.image.convert_image_dtype(img, tf.float32)

  shape = tf.cast(tf.shape(img)[:-1], tf.float32)
  long_dim = max(shape)
  scale = max_dim / long_dim

  new_shape = tf.cast(shape * scale, tf.int32)

  img = tf.image.resize(img, new_shape)
  img = img[tf.newaxis, :]
  return img

In [ ]:
def imshow(image, title=None):
  if len(image.shape) > 3:
    image = tf.squeeze(image, axis=0)
  plt.imshow(image)
  if title:
    plt.title(title)

In [ ]:

import matplotlib.pyplot as plt
content_image = load_img(content_path)
style_image = load_img(style_path)

plt.subplot(1, 2, 1)
imshow(content_image, 'Content Image')

plt.subplot(1, 2, 2)
imshow(style_image, 'Style Image')

In [ ]:
import tensorflow_hub as hub
hub_model = hub.load('https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2')
stylized_image = hub_model(tf.constant(content_image), tf.constant(style_image))[0]
tensor_to_image(stylized_image)